# Задача 9. Markov Chain Monte-Carlo (PyMC)

**Что нужно**
1. Реализовать модель из Задач 1–3 в `pymc`.
2. Получить отклик детектора (распределение полного числа фотоэлектронов) для источника 1 МэВ в центре и сравнить с распределением из Задачи 4.
3. (Опционально) добавить поглощение с параметрами Задачи 6: $\lambda=8$ м.

Установка: `pip install pymc`. Детектор — как в Задаче 4: $R=2$ м, $N=40$, $r=10$ см, $a=0.5$, $Y=10\,000$ фотонов/МэВ.

## Вероятностная модель

Цепочка Задач 1–3 для источника в центре:

$$n\sim\mathrm{Pois}(YE)\;\to\;\text{каждый фотон идёт в ФЭУ } i \text{ с вероятностью } p_i\;\to\;q_i\mid h_i\sim\mathrm{Pois}(a\,h_i),\qquad Q=\sum_i q_i.$$

Здесь $p_i=(1-\cos\alpha)/2$ — доля телесного угла, закрытая шапкой одного ФЭУ (в центре она одинакова для всех ФЭУ, $\sum_i p_i=f$ — покрытие).

**Прореживание Пуассона.** Если $n\sim\mathrm{Pois}(\mu)$ и каждый из $n$ объектов независимо попадает в «корзину» $i$ с вероятностью $p_i$, то числа в корзинах $h_i$ — **независимые** $\mathrm{Pois}(\mu p_i)$. Поэтому модель записывается без дискретного родителя $n$:

$$h_i\sim\mathrm{Pois}(Y E\,p_i),\qquad q_i\sim\mathrm{Pois}(a\,h_i),\qquad Q=\sum_i q_i.$$

Это в точности та же совместная модель, но в виде простого графа из 40 пар пуассоновских величин.

**Поглощение (Задача 6).** Свет ослабляется как $T(d)=e^{-d/\lambda}$. Из центра до стенки $d=R$ для всех фотонов, поэтому каждый фотон выживает с вероятностью $T=e^{-R/\lambda}$, и $p_i\to p_iT$ (снова прореживание).

**Про «MCMC».** В модели нет наблюдаемых данных, поэтому `pm.sample_prior_predictive` делает прямую (ancestral) выборку: независимые точные розыгрыши из модели, без цепи Маркова. Цепь Маркова (NUTS/Metropolis) понадобилась бы для апостериорного вывода, например оценки $E$ по измеренному $Q$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import pymc as pm

from detector import Detector
from source import Source

plt.rcParams['figure.figsize'] = (6, 3)
plt.rcParams['figure.autolayout'] = True

det = Detector(R=2.0, N=40, r=0.10, a=0.5, seed=1)
src = Source(Y=10_000, seed=2)
print(det, src)

E = 1.0                      # МэВ
CENTER = np.zeros(3)
LAMBDA = 8.0                 # м, длина затухания из Задачи 6
N_EVENTS = 1000             # как в Задаче 4
N_DRAWS = 20_000            # розыгрышей из модели PyMC

## 1. Геометрия → вероятности попадания $p_i$

Проверим формулу $p_i=(1-\cos\alpha)/2$ прямым трассированием фотонов из `Detector.trace`.

In [ ]:
p_pmt = np.full(det.N, (1 - np.cos(det.alpha)) / 2)
print(f"p_i = {p_pmt[0]:.6f},  сумма (покрытие) = {p_pmt.sum():.5f}  (Detector.coverage = {det.coverage:.5f})")

n_total, chunk = 1_000_000, 100_000
cnt = np.zeros(det.N)
for _ in range(n_total // chunk):
    idx = det.trace(CENTER, src.random_directions(chunk))
    cnt += np.bincount(idx[idx >= 0], minlength=det.N)

z = (cnt / n_total - p_pmt) / np.sqrt(p_pmt / n_total)
print(f"Монте-Карло по каналам: max |z| = {np.abs(z).max():.2f}, среднее z = {z.mean():+.2f}  (ожидается |z| ≲ 3)")
assert np.abs(z).max() < 4.5

## 2. Модель в PyMC

In [ ]:
def build_model(p, Y, E, a, T=1.0):
    """h_i ~ Pois(Y E T p_i);  q_i ~ Pois(a h_i);  Q = sum q_i.   T — пропускание света."""
    with pm.Model() as model:
        hits = pm.Poisson("hits", mu=Y * E * T * p, shape=p.size)   # фотоны на фотокатоде ФЭУ i
        pe = pm.Poisson("pe", mu=a * hits, shape=p.size)            # фотоэлектроны ФЭУ i
        pm.Deterministic("H", hits.sum())                           # всего попаданий
        pm.Deterministic("Q", pe.sum())                             # полный заряд события
    return model

def sample_response(model, draws, seed):
    with model:
        idata = pm.sample_prior_predictive(draws=draws, random_seed=seed)
    return idata.prior["Q"].values.ravel(), idata.prior["H"].values.ravel()

model = build_model(p_pmt, src.Y, E, det.a)
print(model)
Q_pm, H_pm = sample_response(model, N_DRAWS, seed=10)
print(Q_pm.shape, Q_pm[:10])

## 3. Сравнение с Задачей 4

Берём тот же метод, что в Задаче 4: `run_events` — явное моделирование каждого фотона через `Source` и `Detector`, 1000 событий.

In [ ]:
def run_events(det, src, E, origin, n_events):
    """n_events событий с энергией E в точке origin; полный заряд каждого (как в Задаче 4)."""
    Q = np.empty(n_events, dtype=np.int64)
    for k in range(n_events):
        Q[k] = det.detect(origin, src.generate(E))[1].sum()
    return Q

Q_t4 = run_events(det, src, E, CENTER, N_EVENTS)

mu_H = src.Y * E * det.coverage
m_th, s_th = det.a * mu_H, np.sqrt(det.a * (1 + det.a) * mu_H)

def row(name, Q):
    print(f"{name:28s} <Q> = {Q.mean():7.2f} ± {Q.std(ddof=1)/np.sqrt(len(Q)):4.2f}   "
          f"sigma = {Q.std(ddof=1):6.2f}   sigma/<Q> = {Q.std(ddof=1)/Q.mean():.4f}")

row("PyMC (20 000 розыгрышей)", Q_pm)
row("Задача 4 (1000 событий)", Q_t4)
print(f"{'теория':28s} <Q> = {m_th:7.2f}              sigma = {s_th:6.2f}   sigma/<Q> = {s_th/m_th:.4f}")

ks = stats.ks_2samp(Q_pm, Q_t4)
print(f"\nДвухвыборочный тест Колмогорова-Смирнова: D = {ks.statistic:.4f}, p = {ks.pvalue:.3f}")

bins = np.arange(min(Q_pm.min(), Q_t4.min()), max(Q_pm.max(), Q_t4.max()) + 2, 2) - 0.5
plt.hist(Q_pm, bins=bins, density=True, alpha=0.5, label='PyMC')
plt.hist(Q_t4, bins=bins, density=True, alpha=0.5, label='Задача 4')
x = np.linspace(bins[0], bins[-1], 300)
plt.plot(x, 2*np.exp(-0.5*((x-m_th)/s_th)**2)/(s_th*np.sqrt(2*np.pi)), 'k--', lw=1,
         label=f'N({m_th:.0f}, {s_th:.1f}) (шаг корзины 2)')
plt.xlabel('Полный заряд Q, ф.э.'); plt.ylabel('Плотность'); plt.title('E = 1 МэВ, источник в центре')
plt.legend(); plt.show()

# проверки
assert abs(Q_pm.mean() - m_th) < 5 * s_th / np.sqrt(N_DRAWS)
assert abs(Q_pm.std(ddof=1) / s_th - 1) < 5 / np.sqrt(2 * (N_DRAWS - 1))
assert abs(Q_pm.mean() - Q_t4.mean()) < 5 * np.hypot(Q_pm.std(), Q_t4.std()) / np.sqrt(N_EVENTS)
assert ks.pvalue > 0.001
print("OK  PyMC и Задача 4 согласуются между собой и с теорией")

## 4. Поглощение света (опционально, параметры Задачи 6)

Для сравнения — «честный» Монте-Карло, как в Задаче 4, но каждый фотон по пути до стенки длиной $L$ выживает с вероятностью $e^{-L/\lambda}$ (то есть закон Бугера–Ламберта–Бера из Задачи 6).

In [ ]:
T = np.exp(-det.R / LAMBDA)       # из центра до стенки любой фотон проходит R
print(f"lambda = {LAMBDA} м, путь = {det.R} м, T = exp(-R/lambda) = {T:.4f}")

model_att = build_model(p_pmt, src.Y, E, det.a, T=T)
Q_pm_att, _ = sample_response(model_att, N_DRAWS, seed=11)

rng_abs = np.random.default_rng(12)

def run_events_attenuated(det, src, E, origin, n_events, lam):
    Q = np.empty(n_events, dtype=np.int64)
    for k in range(n_events):
        d = src.generate(E)
        L = np.linalg.norm(det.intersect_sphere(origin, d) - origin, axis=1)   # длина пути до стенки
        d = d[rng_abs.random(len(d)) < np.exp(-L / lam)]                       # выжившие фотоны
        Q[k] = det.detect(origin, d)[1].sum()
    return Q

Q_att_ref = run_events_attenuated(det, src, E, CENTER, N_EVENTS, LAMBDA)

m_att, s_att = det.a * mu_H * T, np.sqrt(det.a * (1 + det.a) * mu_H * T)
row("PyMC, с поглощением", Q_pm_att)
row("Прямой МК, с поглощением", Q_att_ref)
print(f"{'теория':28s} <Q> = {m_att:7.2f}              sigma = {s_att:6.2f}   sigma/<Q> = {s_att/m_att:.4f}")
print(f"\n<Q>(с) / <Q>(без) = {Q_pm_att.mean()/Q_pm.mean():.4f}   (T = {T:.4f})")
ks2 = stats.ks_2samp(Q_pm_att, Q_att_ref)
print(f"KS (PyMC vs прямой МК): D = {ks2.statistic:.4f}, p = {ks2.pvalue:.3f}")

bins = np.arange(min(Q_pm_att.min(), Q_pm.min()), Q_pm.max() + 2, 2) - 0.5
plt.hist(Q_pm, bins=bins, density=True, alpha=0.5, label='без поглощения')
plt.hist(Q_pm_att, bins=bins, density=True, alpha=0.5, label=f'с поглощением, λ = {LAMBDA:.0f} м')
plt.hist(Q_att_ref, bins=bins, density=True, histtype='step', color='k', label='прямой МК (с поглощением)')
plt.xlabel('Полный заряд Q, ф.э.'); plt.ylabel('Плотность'); plt.title('E = 1 МэВ, источник в центре')
plt.legend(); plt.show()

assert abs(Q_pm_att.mean() - m_att) < 5 * s_att / np.sqrt(N_DRAWS)
assert abs(Q_pm_att.std(ddof=1) / s_att - 1) < 5 / np.sqrt(2 * (N_DRAWS - 1))
assert abs(Q_pm_att.mean() - Q_att_ref.mean()) < 5 * np.hypot(Q_pm_att.std(), Q_att_ref.std()) / np.sqrt(N_EVENTS)
assert ks2.pvalue > 0.001
print("OK  поглощение: PyMC согласуется с прямым Монте-Карло и теорией")

## Выводы

* **Модель.** Цепочка «число фотонов → попадания в ФЭУ → фотоэлектроны» записана в `pymc` через прореживание Пуассона: $h_i\sim\mathrm{Pois}(YEp_i)$, $q_i\sim\mathrm{Pois}(ah_i)$, $Q=\sum q_i$ с $p_i=(1-\cos\alpha)/2$; $p_i$ проверены прямым трассированием фотонов детектора.
* **Отклик 1 МэВ в центре.** Распределение $Q$ из PyMC совпадает с распределением Задачи 4 (близкие $\langle Q\rangle\approx125$ ф.э. и $\sigma\approx13.7$, тест Колмогорова-Смирнова не отвергает совпадение) и с теорией: $\langle Q\rangle=a\,Y E f=125.1$, $\sigma=\sqrt{a(1+a)YEf}=13.7$.
* **Поглощение.** При $\lambda=8$ м и пути $R=2$ м свет пропускается с $T=e^{-R/\lambda}=0.78$: $\langle Q\rangle$ падает до $\approx97$ ф.э. (в $T$ раз), $\sigma$ — до $\approx12.1$ (как $\sqrt T$), относительная ширина растёт с $0.110$ до $0.124$ (в $1/\sqrt T$ раз). Результат PyMC совпадает с прямым моделированием.
* **Про MCMC.** Здесь нужна только прямая выборка из модели. Цепи Маркова понадобятся при обратной задаче — вывод энергии $E$ по измеренному заряду.